In [ ]:
# ─────────────────────────────────────────────
# Cell 1 — Install dependencies
# ─────────────────────────────────────────────
!pip install -q openai openpyxl pandas rdflib

In [ ]:
# ─────────────────────────────────────────────
# Cell 2 — Configuration
# ─────────────────────────────────────────────
import os

GROQ_API_KEY = "..."          # ← paste your Groq API key here
MODEL_ID     = "openai/gpt-oss-120b"
TEMPERATURE  = 0.0         # paper used temp=0 for determinism

In [ ]:
# ─────────────────────────────────────────────
# Cell 3 — Upload files
# ─────────────────────────────────────────────
from google.colab import files

print("Upload Dataset.xlsx")
files.upload()

print("\nUpload patterns.csv (Cancel if you don't have it)")
try:
    up = files.upload()
    HAS_PATTERNS = bool(up)
except Exception:
    HAS_PATTERNS = False

print(f"Patterns loaded: {HAS_PATTERNS}")

Upload Dataset.xlsx


Saving Dataset.xlsx to Dataset (7).xlsx

Upload patterns.csv (Cancel if you don't have it)


Patterns loaded: False


In [ ]:
# ─────────────────────────────────────────────
# Cell 4 — Load Dataset.xlsx
# ─────────────────────────────────────────────
import pandas as pd

xl = pd.ExcelFile("Dataset.xlsx")
print("Sheets:", xl.sheet_names)

df_cqs = xl.parse(xl.sheet_names[0])
print("\nColumns:", list(df_cqs.columns))
print(df_cqs.head(3))

Sheets: ['CQs', 'Story']

Columns: ['StoryID', 'CQID', 'CQText', 'Category of CQ']
  StoryID     CQID                                             CQText  \
0     NaN  JAMSCQ1  What is the content of the observations contai...   
1     NaN  JAMSCQ2           Who is the composer of a musical object?   
2     NaN  JAMSCQ3          Who is the performer of a musical object?   

  Category of CQ  
0            NaN  
1            NaN  
2            NaN  


In [ ]:
# ─────────────────────────────────────────────
# Cell 5 — Filter music rows (CQID must contain "Hospital")
# ─────────────────────────────────────────────

cq_col    = next(c for c in df_cqs.columns if "cq" in c.lower() and "id" in c.lower())
story_col = next(c for c in df_cqs.columns if "story" in c.lower())
text_col  = next(c for c in df_cqs.columns if c.lower() == "cqtext")

print(f"CQID column   : '{cq_col}'")
print(f"StoryID column: '{story_col}'")
print(f"CQ text column: '{text_col}'")
print(f"\nAll CQID values:\n{df_cqs[cq_col].astype(str).unique()}")

mask      = df_cqs[cq_col].astype(str).str.upper().str.contains("HOSPITAL")
music_cqs = df_cqs[mask].reset_index(drop=True)

print(f"\n{len(music_cqs)} hospital CQs found:")
print(music_cqs[[story_col, cq_col, text_col]])

CQID column   : 'CQID'
StoryID column: 'StoryID'
CQ text column: 'CQText'

All CQID values:
['JAMSCQ1' 'JAMSCQ2' 'JAMSCQ3' 'JAMSCQ4' 'JAMSCQ5' 'JAMSCQ6' 'JAMSCQ7'
 'JAMSCQ8' 'JAMSCQ9' 'JAMSCQ10' 'JAMSCQ11' 'JAMSCQ12' 'JAMSCQ13'
 'MPROJCQ1' 'MPROJCQ2' 'MPROJCQ3' 'SOURCECQ1' 'SOURCECQ2' 'SOURCECQ4'
 'SOURCECQ5' 'SOURCECQ6' 'MINSTCQ1' 'MINSTCQ2' 'MINSTCQ3' 'MINSTCQ4'
 'MINSTCQ5' 'MINSTCQ7' 'MROCQ1' 'MROCQ2' 'MROCQ3' 'MROCQ4' 'MROCQ5'
 'MROCQ6' 'MROCQ7' 'MROCQ8' 'MROCQ9' 'MROCQ10' 'PCORECQ1' 'PCORECQ2'
 'PCORECQ3' 'PCORECQ4' 'PCORECQ5' 'PCORECQ6' 'PCORECQ7' 'PCORECQ8'
 'PCORECQ9' 'PCORECQ10' 'PCORECQ11' 'PCORECQ12' 'WHOWHYCQ1' 'WHOWHYCQ4'
 'nan' 'cq2' 'cq4' 'cq6' 'cq7' 'cq8' 'cq9' 'cq11' 'cq12' 'cq13' 'cq14'
 'cq15' 'cq17' 'cq19' 'cq20' 'cq21' 'cq22' 'FestSCQ1' 'FestSCQ2'
 'FestSCQ3' 'FestSCQ4' 'FestSCQ5' 'FestSCQ6' 'FestSCQ7' 'FestSCQ8'
 'FestSCQ9' 'FestSCQ10' 'FestSCQ11' 'FestSCQ12' 'FestSCQ13' 'FestSCQ14'
 'FestSCQ15' 'MusicSCQ1' 'MusicSCQ2' 'MusicSCQ3' 'MusicSCQ4' 'MusicSCQ5'
 'MusicSC

In [ ]:
# Cell 6 — Load stories
story_lookup = {}

if len(xl.sheet_names) > 1:
    df_stories = xl.parse(xl.sheet_names[1])
    sid_col    = next((c for c in df_stories.columns if "story" in c.lower() and "id" in c.lower()), df_stories.columns[0])
    stext_col  = next((c for c in df_stories.columns if c != sid_col), df_stories.columns[1])
    story_lookup = dict(zip(df_stories[sid_col].astype(str), df_stories[stext_col].astype(str)))

    # Only show hospital story
    hospital_story_ids = music_cqs[story_col].astype(str).unique()
    print(f"Stories used by hospital CQs: {hospital_story_ids}")
    for sid in hospital_story_ids:
        text = story_lookup.get(sid, "NOT FOUND")
        print(f"\n--- {sid} ---\n{text[:300]}...")
else:
    print("No second sheet found")

Stories used by hospital CQs: ['HospitalS']

--- HospitalS ---
Hospital setting
Pasquale Di Gennaro first studied to become a nursing assistant, but after working some years he decided to continue studying to become a certified nurse. He took his degree in May 2001. On September 21 (2001) he was employed at Ospedale Riunito delle Tre Valli in the city of Nocera...


In [ ]:
# ─────────────────────────────────────────────
# Cell 7 — Load ontology design patterns (optional)
# ─────────────────────────────────────────────
import json

if HAS_PATTERNS:
    df_pat = pd.read_csv("patterns.csv")
    patterns_json = json.dumps(
        {row["Name"]: row["Pattern_owl"] for _, row in df_pat.iterrows()}
    )
    print(f"Loaded {len(df_pat)} patterns")
else:
    patterns_json = "{}"
    print("No patterns file — running without ODP examples")

No patterns file — running without ODP examples


In [ ]:
# Cell 8
PROMPT_TEMPLATE = """\
Your task is to contribute to creating a piece of well-structured ontology by reading information that appeared in the given story, requirements, and restrictions (if there are any).

The way you approach this is first you pick this competency question "{CQ}" and read the given turtle RDF to know what is the current ontology till this stage (it can be empty at the beginning). Then you add or change the RDF so it can answer this competency question. Your output at each stage is an append to the previous ones, just do not repeat. You only need to solve the question number so do not touch the next questions since they belong to the next stages of development.

Classes are the keywords/classes that are going to be node types in the knowledge graph ontology. Try to extract all classes; in addition, classes can also be defined for reification. We use Turtle Syntax for representation. Hierarchies are rdfs:subClassOf in the turtle syntax. They can be used to classify similar classes in one superclass. Mostly the lengthier the hierarchy the better. Important: Class names have Cl_ as the prefix, for example Cl_Professors. Also keep in mind you can add Equivalent To, General class axioms, Disjoint with, and Disjoint Union of, for each class.

In your ontology modeling, for each competency question, when faced with complex scenarios that involve more than two entities or a combination of entities and datatypes, apply reification. Specifically, create a pivot class to act as an intermediary for these entities, ensuring the nuanced relationships are accurately captured.

Then you need to create properties (owl:Property). In this step, you use classes from the previous stage and create object and data properties to connect them and establish the ontology. Always output a turtle syntax. Data properties are between classes and data types such as xsd:string, xsd:integer, xsd:decimal, xsd:dateTime, xsd:date, xsd:time, xsd:boolean. Object properties are between classes. Feel free to use rdfs:subPropertyOf for creating hierarchies for relations. For modeling properties use these relation characteristics when applicable: Functional, Inverse functional, Transitive, Symmetric, Asymmetric, Reflexive, Irreflexive.

Upon implementation of restrictions, feel free to use owl:equivalentClass with owl:Restriction, owl:allValuesFrom, owl:someValuesFrom, and general class axioms.

Use these prefixes:
@prefix : <http://www.example.org/ontology/hospital#> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .

Important: before writing the OWL code, write this text: "is this competency question answerable by the previous version of the RDF (given below) or not?" (most likely not). If no, write a reification class for this question if needed, then solve it. If it was answerable, simply rewrite the given RDF in the output without changing it.

Here is the story:
{story}
End of story

Here is the last RDF:
{rdf}"""


def build_prompt(cq: str, story: str) -> str:
    return PROMPT_TEMPLATE.format(CQ=cq, story=story, rdf="")

In [ ]:
# ─────────────────────────────────────────────
# Cell 9 — LLM client + call with continuation
# ─────────────────────────────────────────────
from openai import OpenAI
import re

client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=GROQ_API_KEY,
)

def is_truncated(text: str) -> bool:
    return not bool(re.search(r'[.\]}>]\s*$', text.rstrip()))

def call_llm(prompt: str, max_continuations: int = 3) -> str:
    messages = [{"role": "user", "content": prompt}]
    full_output = ""

    for round_num in range(1 + max_continuations):
        response = client.chat.completions.create(
            model=MODEL_ID,
            messages=messages,
            temperature=TEMPERATURE,
            max_tokens=6500,
        )
        chunk = response.choices[0].message.content.strip()
        stop_reason = response.choices[0].finish_reason
        full_output += ("\n" if full_output else "") + chunk

        if stop_reason == "stop" or not is_truncated(full_output):
            break

        if round_num < max_continuations:
            print(f"    ↻ Truncated — continuing (round {round_num + 1})")
            messages.append({"role": "assistant", "content": chunk})
            messages.append({
                "role": "user",
                "content": (
                    "The Turtle output was cut off. Continue exactly from where you stopped. "
                    "No repeated prefixes or triples. Output only the remaining Turtle."
                )
            })

    return full_output

In [ ]:
# ─────────────────────────────────────────────
# Cell 10 — Extract Turtle from LLM response
# ─────────────────────────────────────────────
def extract_turtle(text: str) -> str:
    match = re.search(r"```(?:turtle|ttl)?\s*([\s\S]+?)```", text, re.IGNORECASE)
    if match:
        return match.group(1).strip()
    # If no fences, strip the reasoning preamble and return from first @prefix
    prefix_start = text.find("@prefix")
    if prefix_start != -1:
        return text[prefix_start:].strip()
    return text.strip()

In [ ]:
# ─────────────────────────────────────────────
# Cell 11 — Main loop (one independent call per CQ)
# ─────────────────────────────────────────────
import time, os

os.makedirs("modules", exist_ok=True)

modules = []   # list of (cq_id, ttl_string)

for idx, row in music_cqs.iterrows():
    cq_id   = str(row[cq_col])
    cq_text = str(row[text_col])
    sid     = str(row[story_col])
    story   = story_lookup.get(sid, "")

    print(f"\n{'='*60}")
    print(f"[{idx+1}/{len(music_cqs)}] {cq_id}")
    print(f"  CQ: {cq_text[:100]}...")

    prompt = build_prompt(cq=cq_text, story=story)

    try:
        raw = call_llm(prompt)
        ttl = extract_turtle(raw)
        modules.append((cq_id, ttl))

        # Save individual module
        path = f"modules/{cq_id}.ttl"
        with open(path, "w", encoding="utf-8") as f:
            f.write(ttl)

        print(f"  ✓ Saved {path} ({len(ttl)} chars)")
    except Exception as e:
        print(f"  ✗ Error: {e}")
        modules.append((cq_id, ""))

    time.sleep(1)

print(f"\nDone. {sum(1 for _, t in modules if t)} / {len(modules)} modules generated.")


[1/15] HospitalSCQ1
  CQ: What medical degrees does a certain person have?...
  ✓ Saved modules/HospitalSCQ1.ttl (4214 chars)

[2/15] HospitalSCQ2
  CQ: During what time period did a certain person study for a specific degree?...
  ✓ Saved modules/HospitalSCQ2.ttl (4098 chars)

[3/15] HospitalSCQ3
  CQ: When was a certain person first employed at a certain hospital?...
  ✓ Saved modules/HospitalSCQ3.ttl (2834 chars)

[4/15] HospitalSCQ4
  CQ: In what city is a certain hospital located?...
  ✓ Saved modules/HospitalSCQ4.ttl (4324 chars)

[5/15] HospitalSCQ5
  CQ: In what country is a certain city located?...
  ✓ Saved modules/HospitalSCQ5.ttl (2601 chars)

[6/15] HospitalSCQ6
  CQ: Who are the members of a certain union at a certain point in time?...
  ✓ Saved modules/HospitalSCQ6.ttl (4722 chars)

[7/15] HospitalSCQ7
  CQ: What role does a certain person have within a certain union group at a certain point in time?...
  ✓ Saved modules/HospitalSCQ7.ttl (4940 chars)

[8/15] HospitalSCQ

In [ ]:
# ─────────────────────────────────────────────
# Cell 12 — Merge all modules with rdflib
# (deduplicates triples that appear in multiple modules)
# ─────────────────────────────────────────────
from rdflib import Graph

merged = Graph()

for cq_id, ttl in modules:
    if not ttl:
        print(f"  Skipping {cq_id} (empty)")
        continue
    try:
        g = Graph()
        g.parse(data=ttl, format="turtle")
        merged += g
        print(f"  Merged {cq_id}: {len(g)} triples")
    except Exception as e:
        print(f"  Parse error in {cq_id}: {e}")

print(f"\nTotal triples in merged ontology: {len(merged)}")

  Merged HospitalSCQ1: 70 triples
  Parse error in HospitalSCQ2: at line 1 of <>:
Bad syntax (expected directive or statement) at ^ in:
"b''^b'is this competency question answerable by the previous versi'..."
  Merged HospitalSCQ3: 57 triples
  Merged HospitalSCQ4: 124 triples
  Merged HospitalSCQ5: 50 triples
  Merged HospitalSCQ6: 91 triples
  Merged HospitalSCQ7: 107 triples
  Merged HospitalSCQ8: 48 triples
  Parse error in HospitalSCQ9: at line 54 of <>:
Bad syntax (expected '.' or '}' or ']' at end of statement) at ^ in:
"...b'rdfs:label "has article"@en ;\n    owl:inverseOf :isArticleOf'^b' .\n\n: isArticleOf a owl:ObjectProperty ;\n    rdfs:domain :Cl'..."
  Merged HospitalSCQ10: 10 triples
  Merged HospitalSCQ11: 37 triples
  Merged HospitalSCQ12: 23 triples
  Merged HospitalSCQ13: 132 triples
  Merged HospitalSCQ14: 71 triples
  Parse error in HospitalSCQ15: at line 1 of <>:
Bad syntax (expected directive or statement) at ^ in:
"b''^b'is this competency question answerable by

In [ ]:
# ─────────────────────────────────────────────
# Cell 13 — Save and download
# ─────────────────────────────────────────────
import zipfile

from google.colab import files

merged_path = "hospital_memless_CQbyCQ_gpt-oss-120b.ttl"
merged.serialize(destination=merged_path, format="turtle")
print(f"Saved: {merged_path}")
files.download(merged_path)

zip_path = "hospital_modules.zip"
with zipfile.ZipFile(zip_path, "w") as zf:
    for cq_id, ttl in modules:
        if ttl:
            zf.writestr(f"{cq_id}.ttl", ttl)
print(f"Saved: {zip_path}")
files.download(zip_path)


Saved: hospital_memless_CQbyCQ_gpt-oss-120b.ttl


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Saved: hospital_modules.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>